In [ ]:
!pip install -q evaluate jiwer datasets torch
!pip install transformers==4.43.3


### 核心概念验证与环境准备
在真实评估模型之前，我们需要安装 Hugging Face 的 `evaluate` 库以及它底层依赖的计算包 jiwer。

In [ ]:
from evaluate import load

# WER评估模块
wer_metric = load("wer")

# 1. 定义标准答案与预测结果
reference = "the cat sat on the mat"
prediction = "the cat sit on the"

# 2. 计算 WER
wer = wer_metric.compute(references=[reference], predictions=[prediction])

print(f"\n参考答案: {reference}")
print(f"模型预测: {prediction}")
print(f"计算出的 WER: {wer:.4f} (即 {wer * 100:.2f}%)")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



参考答案: the cat sat on the mat
模型预测: the cat sit on the
计算出的 WER: 0.3333 (即 33.33%)


### 加载真实数据集
由于 Mozilla Common Voice 13 的迪维希语（Dhivehi，语言代码 `dv`）测试集已经无法使用。
这里使用 PolyAI/minds14 数据集

In [ ]:
from datasets import load_dataset
from huggingface_hub import notebook_login

minds14 = load_dataset("PolyAI/minds14", name="en-US", split="train")

# 【工程优化】：仅截取前 40 条数据进行快速验证
cv_subset = minds14.select(range(40))

print(f"\n成功加载！抽取了 {len(cv_subset)} 条音频数据。")
# 注意：这个数据集的文本列名叫做 'transcription'
print(f"第一条标答文本: {cv_subset[0]['transcription']}")


成功加载！抽取了 40 条音频数据。
第一条标答文本: I would like to set up a joint account with my partner


### 启动 Whisper 批量推理

In [ ]:
import torch
from transformers import pipeline
from tqdm.auto import tqdm

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"正在使用 {device} 加载 Whisper-Small 模型...")

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device
)

all_predictions = []

# 直接提取纯 numpy 数组列表
audio_arrays = [sample["audio"]["array"] for sample in cv_subset]

print("\n开始单条循环语音识别 (彻底避开底层 batch 碰撞 Bug) ...")

# 【工程终极绝招】：放弃 pipeline 脆弱的 batch_size，手动一条条循环！
for audio in tqdm(audio_arrays, total=len(audio_arrays)):
    # 每次只传一条纯数组，绝对不触发它的 DataLoader 维度拼接逻辑！
    prediction = pipe(
        audio,
        max_new_tokens=128,
        generate_kwargs={"task": "transcribe"}
    )
    all_predictions.append(prediction["text"])

print("\n推理完成！查看前两条预测结果：")
print(all_predictions[:2])

正在使用 cuda:0 加载 Whisper-Small 模型...

开始单条循环语音识别 (彻底避开底层 batch 碰撞 Bug) ...


  0%|          | 0/40 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/whisper/generation_whisper.py:483: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



推理完成！查看前两条预测结果：
[' I would like to set up a joint account with my partner. How do I proceed with doing that?', ' Please write a comment to help me to handle my life, and what are they at my fee?']


In [ ]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

# 1. 提取真实标签 (注意：minds14 数据集的文本列名叫 transcription)
raw_references = cv_subset["transcription"]

# 2. 计算【原始 WER】
# evaluate 库计算的结果是小数，乘以 100 变成百分比
wer_raw = 100 * wer_metric.compute(references=raw_references, predictions=all_predictions)

# 3. 文本正则化 (清洗数据)
print("正在执行文本正则化 (去除标点、统一大小写)...")
normalizer = BasicTextNormalizer()

# 对预测结果和标答都使用 OpenAI 的官方清理器进行清洗
all_predictions_norm = [normalizer(pred) for pred in all_predictions]
all_references_norm = [normalizer(ref) for ref in raw_references]

# 4. 过滤掉清洗后变成“空字符串”的标答 (工程中的防御性编程)
# 防止除以 0 的错误发生
valid_preds_norm = []
valid_refs_norm = []
for pred, ref in zip(all_predictions_norm, all_references_norm):
    if len(ref) > 0:
        valid_preds_norm.append(pred)
        valid_refs_norm.append(ref)

# 5. 计算【正则化后的 WER】
wer_norm = 100 * wer_metric.compute(references=valid_refs_norm, predictions=valid_preds_norm)

# 6. 打印最终报告
print("\n=== ASR 模型评估报告 ===")
print(f"原始 WER (未清理标点): {wer_raw:.2f}%")
print(f"正则化 WER (清理后):   {wer_norm:.2f}%")
print("============================")

print(f"清洗前标答: '{raw_references[0]}'")
print(f"清洗后标答: '{all_references_norm[0]}'")

正在执行文本正则化 (去除标点、统一大小写)...

=== ASR 模型评估报告 ===
原始 WER (未清理标点): 63.91%
正则化 WER (清理后):   54.30%
清洗前标答: 'I would like to set up a joint account with my partner'
清洗后标答: 'i would like to set up a joint account with my partner'
